In [1]:
import sympy as sp
from SymEigen import *
import diff_friction
import sys
sys.path.append('../')
from project_dir import backend_source_dir

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction('__host__ __device__')
Gen.DisableLatexComment()

Xn = Eigen.Matrix('Xn', 6, 1)
X = Eigen.Matrix('X', 6, 1)
dX = X - Xn
dfdtau = Eigen.Matrix('dfdtau', 2, 1)

P0 = Xn[0:3, :]
P1 = Xn[3:6, :]

# e_hat can be e_x = [1,0,0]^T or e_y = [0,1,0]^T
e_hat = Eigen.Matrix('e_hat', 3, 1)

n = P1 - P0

n

Matrix([
[-Xn(0) + Xn(3)],
[-Xn(1) + Xn(4)],
[-Xn(2) + Xn(5)]])

In [2]:
e1 = Eigen.Normalize(n.cross(e_hat))
e2 = Eigen.Normalize(n.cross(e1))

T = sp.Matrix.zeros(2, 6)
T[0, 0:3] = e1.T
T[1, 0:3] = e2.T
T[0, 3:6] = -e1.T
T[1, 3:6] = -e2.T

In [3]:
content = ''
Cl = Gen.Closure(Xn, e_hat)
content += Cl('TR', T) + '\n'

In [4]:
dTdXn = diff_friction.compute_dTdXn(T, Xn)
dTdXn.shape

(6, 2, 6)

In [5]:
A, B = diff_friction.compute_AB(dTdXn, dX, dfdtau)
A.shape, B.shape

((2, 6), (6, 6))

In [6]:
Cl = Gen.Closure(Xn, X, e_hat)
content += Cl('An', A) + '\n'

In [7]:
Cl = Gen.Closure(Xn, X, dfdtau, e_hat)
content += Cl('Bn', B) + '\n'

In [8]:
filename = 'ipc_diff_friction_PP'
cuda_backend_source_dir = backend_source_dir('cuda')
f = open(f'{cuda_backend_source_dir}/contact_system/contact_models/sym/{filename}.inl', 'w')
f.write(content)
f.close()